In [1]:
spark

In [10]:
sc.addPyFile('/opt/spark-3.5.1/jars/graphframes-0.8.4-spark3.5-s_2.12.jar')
from pyspark.sql.functions import *

25/11/21 23:46:16 WARN SparkContext: The path /opt/spark-3.5.1/jars/graphframes-0.8.4-spark3.5-s_2.12.jar has been added already. Overwriting of added paths is not supported in the current version.


In [3]:
from graphframes import GraphFrame

In [4]:
# GraphFrames & ALgorithms

In [5]:
#Breadth -first search (BFS) <-- shortest Path from vertix has a specific Description to 
#Vertix has anthor specific description 

In [6]:
#Vertix Dataframe
v = spark.createDataFrame([
    ('a', 'Alice', 34,'DS',3000),
    ('b', 'Bob', 36,'ML',2000),
    ('c', 'Charlie', 30,'WebDev',800),
    ('d', 'David', 29,'Programmer',1000),
    ('e', 'Esther', 32,'None',0),
    ('f', 'Fanny', 36,'ML',20000),
    ('g', 'Gabby', 60,'Retired',5000)
], ['id', 'name', 'age' , 'job','salary'])
#Edge Dataframe
e = spark.createDataFrame([
    ('a', 'b', 'friend',2),
    ('b', 'c', 'follow',6),
    ('c', 'b', 'follow',7),
    ('f', 'c', 'follow',0),
    ('e', 'f', 'follow',1),
    ('e', 'd', 'friend',2),
    ('d', 'a', 'friend',5),
    ('a', 'e', 'friend',9)
], ['src', 'dst', 'relationship', 'strength'])
g = GraphFrame(v,e)

In [9]:
g.vertices.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- job: string (nullable = true)
 |-- salary: long (nullable = true)



In [17]:
paths = g.bfs('name = "Esther"' ,' age < 32 ')

In [18]:
paths.show()

+--------------------+-----------------+--------------------+
|                from|               e0|                  to|
+--------------------+-----------------+--------------------+
|{e, Esther, 32, N...|{e, d, friend, 2}|{d, David, 29, Pr...|
+--------------------+-----------------+--------------------+



In [20]:
paths = g.bfs('age > 32' ,' age < 32 ')
paths.show()

+--------------------+-----------------+--------------------+
|                from|               e0|                  to|
+--------------------+-----------------+--------------------+
|{f, Fanny, 36, ML...|{f, c, follow, 0}|{c, Charlie, 30, ...|
|{b, Bob, 36, ML, ...|{b, c, follow, 6}|{c, Charlie, 30, ...|
+--------------------+-----------------+--------------------+



In [23]:
paths = g.bfs('name = "Esther"' ,' age < 32 ',\
             edgeFilter='relationship != "friend"' , maxPathLength=3)
paths.show()

+--------------------+-----------------+--------------------+-----------------+--------------------+
|                from|               e0|                  v1|               e1|                  to|
+--------------------+-----------------+--------------------+-----------------+--------------------+
|{e, Esther, 32, N...|{e, f, follow, 1}|{f, Fanny, 36, ML...|{f, c, follow, 0}|{c, Charlie, 30, ...|
+--------------------+-----------------+--------------------+-----------------+--------------------+



In [ ]:
# Secomd Algorithm (Shortest Path) 
# find a shortest path from graph to landmarks vertices

In [25]:
results = g.shortestPaths(landmarks=['a','d'])

In [26]:
results.show()

[Stage 276:>                                                        (0 + 1) / 1]

+---+-------+---+----------+------+----------------+
| id|   name|age|       job|salary|       distances|
+---+-------+---+----------+------+----------------+
|  g|  Gabby| 60|   Retired|  5000|              {}|
|  f|  Fanny| 36|        ML| 20000|              {}|
|  e| Esther| 32|      None|     0|{a -> 2, d -> 1}|
|  d|  David| 29|Programmer|  1000|{a -> 1, d -> 0}|
|  c|Charlie| 30|    WebDev|   800|              {}|
|  b|    Bob| 36|        ML|  2000|              {}|
|  a|  Alice| 34|        DS|  3000|{a -> 0, d -> 2}|
+---+-------+---+----------+------+----------------+



In [27]:
# Connected Component Algorithm

In [29]:
spark.sparkContext.setCheckpointDir('chpointDir')
result = g.connectedComponents()

In [30]:
result.show()

+---+-------+---+----------+------+------------+
| id|   name|age|       job|salary|   component|
+---+-------+---+----------+------+------------+
|  a|  Alice| 34|        DS|  3000|412316860416|
|  b|    Bob| 36|        ML|  2000|412316860416|
|  c|Charlie| 30|    WebDev|   800|412316860416|
|  d|  David| 29|Programmer|  1000|412316860416|
|  e| Esther| 32|      None|     0|412316860416|
|  f|  Fanny| 36|        ML| 20000|412316860416|
|  g|  Gabby| 60|   Retired|  5000|146028888064|
+---+-------+---+----------+------+------------+



In [34]:
# Strongly Connected Components (can reach for each vertex)
result =g.stronglyConnectedComponents(maxIter=10)

In [35]:
result.show()

[Stage 3763:============================>                           (2 + 2) / 4]

+---+-------+---+----------+------+-------------+
| id|   name|age|       job|salary|    component|
+---+-------+---+----------+------+-------------+
|  g|  Gabby| 60|   Retired|  5000| 146028888064|
|  f|  Fanny| 36|        ML| 20000| 412316860416|
|  e| Esther| 32|      None|     0| 670014898176|
|  d|  David| 29|Programmer|  1000| 670014898176|
|  c|Charlie| 30|    WebDev|   800|1047972020224|
|  b|    Bob| 36|        ML|  2000|1047972020224|
|  a|  Alice| 34|        DS|  3000| 670014898176|
+---+-------+---+----------+------+-------------+



In [37]:
result.select(col('id'), col('component')).orderBy('component').show()

+---+-------------+
| id|    component|
+---+-------------+
|  g| 146028888064|
|  f| 412316860416|
|  e| 670014898176|
|  d| 670014898176|
|  a| 670014898176|
|  c|1047972020224|
|  b|1047972020224|
+---+-------------+



In [38]:
vert = spark.createDataFrame([
    ('a',1), ('b',2),('c',3),('d',4)
],['id','No'])
Edg = spark.createDataFrame([
    ('a','b','ab'),('b','a','ba'),('c','d','cd'),('d','c','dc'),('b','d','bd'),('c','a','ca')
],['src','dst','TheRelationship'])

In [39]:
g2 = GraphFrame(vert,Edg)
g2.vertices.show()
g2.edges.show()

+---+---+
| id| No|
+---+---+
|  a|  1|
|  b|  2|
|  c|  3|
|  d|  4|
+---+---+

+---+---+---------------+
|src|dst|TheRelationship|
+---+---+---------------+
|  a|  b|             ab|
|  b|  a|             ba|
|  c|  d|             cd|
|  d|  c|             dc|
|  b|  d|             bd|
|  c|  a|             ca|
+---+---+---------------+



In [40]:
result = g2.stronglyConnectedComponents(maxIter=10)
result.show()

+---+---+------------+
| id| No|   component|
+---+---+------------+
|  d|  4|807453851648|
|  c|  3|807453851648|
|  b|  2|807453851648|
|  a|  1|807453851648|
+---+---+------------+



In [ ]:
# Label Propagation Algoritm (LPA)
# propagate a label and discover pattern 

In [41]:
result = g2.labelPropagation(maxIter=10)
result.show()

+---+---+-------------+
| id| No|        label|
+---+---+-------------+
|  d|  4| 807453851648|
|  c|  3|1047972020224|
|  b|  2|1382979469312|
|  a|  1|1460288880640|
+---+---+-------------+



In [44]:
# Page Rank


In [46]:
result = g.pageRank(maxIter=10)

In [50]:
result.vertices.orderBy('pagerank').show()
result.edges.orderBy('weight').show()

+---+-------+---+----------+------+-------------------+
| id|   name|age|       job|salary|           pagerank|
+---+-------+---+----------+------+-------------------+
|  g|  Gabby| 60|   Retired|  5000|0.17073170731707318|
|  f|  Fanny| 36|        ML| 20000|0.32504910549694244|
|  d|  David| 29|Programmer|  1000|0.32504910549694244|
|  e| Esther| 32|      None|     0| 0.3613490987992571|
|  a|  Alice| 34|        DS|  3000| 0.4485115093698443|
|  c|Charlie| 30|    WebDev|   800| 2.6667877057849627|
|  b|    Bob| 36|        ML|  2000| 2.7025217677349773|
+---+-------+---+----------+------+-------------------+



+---+---+------------+--------+------+
|src|dst|relationship|strength|weight|
+---+---+------------+--------+------+
|  e|  f|      follow|       1|   0.5|
|  e|  d|      friend|       2|   0.5|
|  a|  e|      friend|       9|   0.5|
|  a|  b|      friend|       2|   0.5|
|  f|  c|      follow|       0|   1.0|
|  d|  a|      friend|       5|   1.0|
|  c|  b|      follow|       7|   1.0|
|  b|  c|      follow|       6|   1.0|
+---+---+------------+--------+------+



In [51]:
# Triangler Count 
result = g.triangleCount()
result.show()

+-----+---+-------+---+----------+------+
|count| id|   name|age|       job|salary|
+-----+---+-------+---+----------+------+
|    1|  a|  Alice| 34|        DS|  3000|
|    0|  c|Charlie| 30|    WebDev|   800|
|    0|  b|    Bob| 36|        ML|  2000|
|    1|  e| Esther| 32|      None|     0|
|    1|  d|  David| 29|Programmer|  1000|
|    0|  g|  Gabby| 60|   Retired|  5000|
|    0|  f|  Fanny| 36|        ML| 20000|
+-----+---+-------+---+----------+------+

